# 03 · Attention Visualization & XAI

**Purpose**: Inspect what the `AttentionDQNAgent` attends to over time and which features
drive its portfolio decisions.

**Prerequisites**: Run `01_data_regimes.ipynb` and `02_train_evaluate.ipynb` first.

**What this notebook covers**:
1. Load a trained `AttentionDQNAgent` checkpoint
2. Collect attention matrices on a chosen split
3. Plot mean attention heatmap + timestep importance
4. Regime-conditioned attention (query-regime → key-regime)
5. Attention concentration diagnostics
6. Gradient × Input feature attribution (XAI)
7. Action-conditioned attention (does the model attend differently per action?)


## 1 · Imports & Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = None
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "full_pipeline").exists() and (_c / "ml").exists():
        REPO_ROOT = _c
        break
if REPO_ROOT is None:
    raise RuntimeError("Could not locate repo root.")

PIPELINE_ROOT = REPO_ROOT / "full_pipeline"
for _p in (str(REPO_ROOT), str(PIPELINE_ROOT)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from evaluation import EvaluationConfig
from ml.agents import AttentionDQNAgent
from ml.hyperparameter_config import load_hyperparameter_config
from _pipeline_utils import OUTPUT_DIR, make_rl_env, prepare_rl_inputs

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 220)
pd.set_option("display.width", 220)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 2 · Configuration

In [ ]:
CONFIG_PATH = REPO_ROOT / "configs" / "rl_hyperparameters.yaml"
HP_CONFIG   = load_hyperparameter_config(CONFIG_PATH, fast_mode=False)
CFG         = HP_CONFIG.values

SEQ_LEN  = int(CFG["general"]["sequence_length"])
SPLIT    = "locked_test"   # change to "validation" if needed
print(f"SEQ_LEN={SEQ_LEN} | SPLIT={SPLIT!r}")


## 3 · Load RL State

In [ ]:
state_path = OUTPUT_DIR / "model_state_weekly_hmm_news.csv"

def _prepare_resilient(path):
    try:
        return prepare_rl_inputs(state_path=path)
    except ValueError as exc:
        if "week_end" not in str(exc):
            raise
        raw = pd.read_csv(path)
        raw.columns = [str(c).strip() for c in raw.columns]
        cleaned = path.with_name(path.stem + "_cleaned.csv")
        raw.to_csv(cleaned, index=False)
        print("Recovered malformed headers →", cleaned)
        return prepare_rl_inputs(state_path=cleaned)

prepared = _prepare_resilient(state_path)
eval_config = EvaluationConfig(
    transaction_cost = float(CFG["environment"]["transaction_cost"]),
    risk_penalty     = float(CFG["environment"]["volatility_penalty"]),
    risk_window      = int(CFG["environment"]["lookback_vol"]),
)

env = make_rl_env(prepared, split=SPLIT, seq_len=SEQ_LEN, config=eval_config)
state_dim  = int(env.observation_space.shape[-1])
action_dim = int(env.action_space.n)

display(prepared["dataset"].describe_splits())
print(f"split={SPLIT} | state_dim={state_dim} | action_dim={action_dim} | seq_len={SEQ_LEN}")


## 4 · Load AttentionDQN Checkpoint

In [ ]:
agent = AttentionDQNAgent(
    state_dim     = state_dim,
    action_dim    = action_dim,
    seq_len       = SEQ_LEN,
    learning_rate = float(CFG["dqn"]["learning_rate"]),
    gamma         = float(CFG["dqn"]["gamma"]),
    epsilon_start = float(CFG["dqn"]["exploration_initial_eps"]),
    epsilon_end   = float(CFG["dqn"]["exploration_final_eps"]),
    epsilon_decay = 3000,
    buffer_capacity = int(CFG["dqn"]["buffer_size"]),
    batch_size    = int(CFG["dqn"]["batch_size"]),
    target_update_freq = max(250, int(CFG["dqn"]["target_update_interval"] // 4)),
    use_dueling   = True,
    device        = DEVICE,
)

checkpoint_candidates = [
    OUTPUT_DIR / "attention_dqn_hmm_news_finetuned.pt",
    OUTPUT_DIR / "attention_dqn_hmm_news_base.pt",
    REPO_ROOT / "output" / "models" / "attention_dqn_agent.pt",
]

ckpt = next((p for p in checkpoint_candidates if p.exists()), None)
if ckpt is None:
    raise FileNotFoundError(
        "No AttentionDQN checkpoint found. "
        f"Tried: {[str(p) for p in checkpoint_candidates]}"
    )

agent.load_checkpoint(str(ckpt))
print("Loaded:", ckpt.relative_to(REPO_ROOT))


## 5 · Collect Attention Rollout

In [ ]:
def _normalize_attention(att):
    arr = np.asarray(att)
    if arr.ndim == 4: arr = arr[0]
    if arr.ndim == 3: arr = arr.mean(axis=0)
    if arr.ndim != 2:
        raise ValueError(f"Unexpected attention shape: {arr.shape}")
    return arr

n_features   = len(prepared["feature_cols"])
n_regimes    = len(prepared["posterior_cols"])
regime_start = n_features
regime_end   = n_features + n_regimes
regime_labels = [c.replace("filtered_prob_regime_", "R") for c in prepared["posterior_cols"]]

obs, _ = env.reset()
done   = False

attention_mats, observations, actions, rewards, step_idx = [], [], [], [], []
key_regime_sequences = []

step = 0
while not done:
    att     = _normalize_attention(agent.get_attention_weights(obs))
    reg_seq = np.argmax(obs[:, regime_start:regime_end], axis=1).astype(int)

    observations.append(obs.copy())
    key_regime_sequences.append(reg_seq)
    attention_mats.append(att)

    action = agent.select_action(obs, training=False)
    obs, reward, terminated, truncated, _ = env.step(action)
    done = bool(terminated or truncated)

    actions.append(int(action))
    rewards.append(float(reward))
    step_idx.append(step)
    step += 1

obs_stack       = np.stack(observations,   axis=0)   # [T, seq, state]
attention_stack = np.stack(attention_mats, axis=0)   # [T, seq, seq]
mean_attention  = attention_stack.mean(axis=0)
timestep_importance = mean_attention.mean(axis=0)

# Regime-to-regime attention
regime_pair_sum   = np.zeros((n_regimes, n_regimes))
regime_pair_count = np.zeros((n_regimes, n_regimes))
regime_key_rows   = []

for i, att_mat in enumerate(attention_mats):
    seq_reg = key_regime_sequences[i]
    for q in range(att_mat.shape[0]):
        for k in range(att_mat.shape[1]):
            regime_pair_sum  [int(seq_reg[q]), int(seq_reg[k])] += att_mat[q, k]
            regime_pair_count[int(seq_reg[q]), int(seq_reg[k])] += 1.0
    latest = att_mat[-1]
    for kp, w in enumerate(latest):
        regime_key_rows.append({"step": i, "key_pos": kp,
                                 "key_regime_id": int(seq_reg[kp]), "attention": float(w)})

regime_pair_attention = np.divide(regime_pair_sum, np.maximum(regime_pair_count, 1.0))
regime_key_df = pd.DataFrame(regime_key_rows)
regime_importance = (regime_key_df.groupby("key_regime_id")["attention"]
                     .mean().reset_index().sort_values("key_regime_id"))
regime_importance["regime"] = regime_importance["key_regime_id"].map(
    lambda i: regime_labels[int(i)] if int(i) < len(regime_labels) else f"R{i}")

rollout_df = pd.DataFrame({"step": step_idx, "action_id": actions, "reward": rewards})
print(f"Collected {len(step_idx)} steps | obs {obs_stack.shape} | attention {attention_stack.shape}")
display(rollout_df.head())
display(regime_importance)


## 6 · Main Attention Figure

- **Top-left**: Mean temporal attention matrix — query (row) → key (column).
- **Top-right**: Average key-side importance by sequence position.
- **Bottom-left**: Regime-to-regime attention (aggregated over all steps).
- **Bottom-right**: Mean attention received by each regime from the latest query step.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

im = axes[0, 0].imshow(mean_attention, aspect="auto", cmap="viridis")
axes[0, 0].set_title(f"Mean Attention Matrix ({SPLIT})")
axes[0, 0].set_xlabel("Key timestep"); axes[0, 0].set_ylabel("Query timestep")
fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.04)

axes[0, 1].plot(np.arange(len(timestep_importance)), timestep_importance, marker="o", lw=2)
axes[0, 1].set_title("Average Attention by Timestep Position")
axes[0, 1].set_xlabel("Timestep in sequence"); axes[0, 1].set_ylabel("Weight")
axes[0, 1].grid(True, alpha=0.3)

im2 = axes[1, 0].imshow(regime_pair_attention, aspect="auto", cmap="magma")
axes[1, 0].set_title("Regime-to-Regime Attention")
axes[1, 0].set_xlabel("Key regime"); axes[1, 0].set_ylabel("Query regime")
axes[1, 0].set_xticks(np.arange(n_regimes)); axes[1, 0].set_xticklabels(regime_labels, rotation=30, ha="right")
axes[1, 0].set_yticks(np.arange(n_regimes)); axes[1, 0].set_yticklabels(regime_labels)
fig.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

axes[1, 1].bar(regime_importance["regime"], regime_importance["attention"], color="teal")
axes[1, 1].set_title("Attention to Key Regime (latest query step)")
axes[1, 1].set_xlabel("Regime"); axes[1, 1].set_ylabel("Mean weight")
axes[1, 1].tick_params(axis="x", rotation=30); axes[1, 1].grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()


## 7 · Attention Concentration Diagnostics

Checks whether attention is informative or near-uniform.

| Metric | Interpretation |
|---|---|
| `avg_norm_entropy` > 0.95 | Very diffuse — near-uniform attention |
| `avg_norm_entropy` < 0.85 | Meaningful concentration |
| `avg_top1_share` | Fraction absorbed by single key per step |


In [ ]:
latest_att = attention_stack[:, -1, :]          # [T, seq_len]
uniform    = np.full(latest_att.shape[1], 1.0 / latest_att.shape[1])

l1_dist  = np.abs(latest_att - uniform).sum(axis=1)
eps      = 1e-12
entropy  = -(latest_att * np.log(latest_att + eps)).sum(axis=1)
norm_ent = entropy / np.log(latest_att.shape[1])
top1     = latest_att.max(axis=1)

display(pd.DataFrame({
    "metric": ["min_weight", "max_weight", "mean_weight", "std_weight",
               "uniform_weight", "avg_l1_to_uniform", "avg_norm_entropy", "avg_top1_share"],
    "value":  [latest_att.min(), latest_att.max(), latest_att.mean(), latest_att.std(),
               float(uniform[0]), float(l1_dist.mean()),
               float(norm_ent.mean()), float(top1.mean())],
}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(l1_dist, bins=20, color="slateblue", alpha=0.85)
axes[0].set_title("L1 Distance to Uniform"); axes[0].set_xlabel("L1 distance")
axes[1].hist(top1, bins=20, color="darkorange", alpha=0.85)
axes[1].set_title("Top-1 Attention Share"); axes[1].set_xlabel("Top-1 share")
fig.tight_layout(); plt.show()

if norm_ent.mean() > 0.95:
    print("Attention is very diffuse (near-uniform).")
elif norm_ent.mean() > 0.85:
    print("Attention is moderately diffuse.")
else:
    print("Attention shows meaningful concentration.")


## 8 · Gradient × Input Feature Attribution (XAI)

For each rollout step, computes |∂Q(a) / ∂x · x| — absolute Gradient × Input — as a
per-feature importance score. Aggregated by:
- **Semantic group** (price, macro, text, regime, prev_alloc) × timestep position
- **Top individual features** at the latest timestep

High regime-group attribution → agent is regime-sensitive.  
High prev_alloc attribution → agent is path-dependent.


In [ ]:
was_training = agent.q_network.training
agent.q_network.train()

feature_names = list(prepared["feature_cols"]) + regime_labels + [
    "prev_alloc_spy", "prev_alloc_tlt", "prev_alloc_gld", "prev_alloc_cash"]

n_price = len(prepared["dataset"].feature_groups.price)
n_macro = len(prepared["dataset"].feature_groups.macro)
n_text  = len(prepared["dataset"].feature_groups.text)

group_slices = {
    "price":     slice(0, n_price),
    "macro":     slice(n_price, n_price + n_macro),
    "text":      slice(n_price + n_macro, n_features),
    "regime":    slice(regime_start, regime_end),
    "prev_alloc": slice(regime_end, regime_end + 4),
}

attr_tensors = []
for i in range(obs_stack.shape[0]):
    s_t = torch.tensor(obs_stack[i], dtype=torch.float32, device=agent.device).unsqueeze(0)
    s_t.requires_grad_(True)
    q_vals, _ = agent.q_network(s_t)
    agent.q_network.zero_grad(set_to_none=True)
    q_vals[0, actions[i]].backward()
    grad = s_t.grad.detach().cpu().numpy()[0]
    attr_tensors.append(np.abs(grad * obs_stack[i]))

attr_stack = np.stack(attr_tensors, axis=0)   # [T, seq, state_dim]
mean_attr  = attr_stack.mean(axis=0)           # [seq, state_dim]

group_ts_rows, group_global = [], []
for gname, sl in group_slices.items():
    vals = mean_attr[:, sl].sum(axis=1)
    group_global.append({"group": gname, "attribution": float(vals.mean())})
    for t, v in enumerate(vals):
        group_ts_rows.append({"timestep_pos": t, "group": gname, "attribution": float(v)})

group_global_df = pd.DataFrame(group_global).sort_values("attribution", ascending=False)
group_ts_df     = pd.DataFrame(group_ts_rows)

top_k      = min(15, len(feature_names))
latest_attr = mean_attr[-1]
top_idx     = np.argsort(latest_attr)[-top_k:][::-1]
top_feature_df = pd.DataFrame({
    "feature":     [feature_names[j] for j in top_idx],
    "attribution": [float(latest_attr[j]) for j in top_idx],
})

display(group_global_df)
display(top_feature_df)

groups = list(group_slices.keys())
heat   = np.zeros((SEQ_LEN, len(groups)))
for gi, gname in enumerate(groups):
    vals = group_ts_df.loc[group_ts_df["group"] == gname, "attribution"].to_numpy()
    heat[:len(vals), gi] = vals

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
im = axes[0].imshow(heat, aspect="auto", cmap="cividis")
axes[0].set_title("Group Attribution by Timestep (Grad × Input)")
axes[0].set_xlabel("Feature group"); axes[0].set_ylabel("Timestep pos")
axes[0].set_xticks(np.arange(len(groups)))
axes[0].set_xticklabels(groups, rotation=20, ha="right")
axes[0].set_yticks(np.arange(SEQ_LEN))
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04)

axes[1].barh(top_feature_df["feature"][::-1], top_feature_df["attribution"][::-1], color="seagreen")
axes[1].set_title("Top Feature Attributions (latest timestep)")
axes[1].set_xlabel("Attribution"); axes[1].set_ylabel("Feature")
axes[1].grid(axis="x", alpha=0.25)
fig.tight_layout(); plt.show()

if not was_training:
    agent.q_network.eval()


## 9 · Action-Conditioned Attention

Compares temporal attention profiles for each portfolio action selected during the rollout.
A clear shift between risk-on and defensive actions implies regime-aware attention.


In [ ]:
action_rows = []
for aid in sorted(set(actions)):
    idx = [i for i, a in enumerate(actions) if a == aid]
    if not idx:
        continue
    mean_by_action = attention_stack[idx].mean(axis=0).mean(axis=0)
    for pos, val in enumerate(mean_by_action):
        action_rows.append({"action_id": aid, "timestep_pos": pos, "attention": float(val)})

action_att_df = pd.DataFrame(action_rows)
action_names  = getattr(env, "ACTION_NAMES", {})
display(action_att_df.head(20))

if not action_att_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    for aid, grp in action_att_df.groupby("action_id"):
        label = action_names.get(int(aid), f"action {aid}") if isinstance(action_names, dict) else f"action {aid}"
        ax.plot(grp["timestep_pos"], grp["attention"], marker="o", label=label)
    ax.set_title("Attention by Timestep — per Action")
    ax.set_xlabel("Timestep position"); ax.set_ylabel("Attention weight")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    fig.tight_layout(); plt.show()
